# Week 6 on Kaggle: real VoIP-app transmission (RTCFake)

RTCFake (Xue et al., ACL Findings 2026) sent each clip through real communication apps (Zoom, QQ, WeChat in its dev split) and kept the **offline original paired with the received online copy**. This notebook uses the **English dev pairs** (genuine and spoofed) with frozen checkpoints A, B and XLSR-SLS:

1. **W6-1** ASVspoof 2019 clean pass (to rebuild each detector's clean real-to-fake axis) and the week-4 drift-to-rejection curves.
2. **W6-2** Download RTCFake dev (2.7 GB), keep the English pairs, score offline and online copies and record all 24 layers.
3. **W6-3** Does real transmission push genuine speech along the same axis? Rejection offline vs online, genuine drift per layer, and the prediction from the curves fitted on simulated channels.
4. **W6-4** Within the channel: do the utterances that drift more lose more score?
5. **W6-5** Correction on real transmission (checkpoint A): fit the mean drift on half of the pairs, remove it at layer 17 on the other half.
6. **W6-6** Summary for Claude and the zip for GitHub.

**Before you run:** GPU **T4 x2**, **Internet On**; inputs as in week 5 (Drive link already filled; add `kithminadev/ssl-aasist-checkpoint-b`); **Add-ons → Secrets: `HF_TOKEN`** switched on for this notebook (your Hugging Face read token, with RTCFake access accepted). Then **Save Version → Save & Run All**. About 1.5 hours. Send Claude `week6_summary.txt`.

In [ ]:
#@title 0 · Kaggle: fetch your data and lay it out like the Drive folder
DRIVE_ZIP_LINK = 'https://drive.google.com/file/d/1pzQ1AoDFrY682zaKn8lEorvy7sv8vE7Z/view?usp=sharing'   # paste the Google Drive share link of fyp_kaggle.zip here (option A), or leave '' if you added a Kaggle Dataset (option B)

import os, glob, shutil, subprocess
OUTP = '/kaggle/working/fyp_pilot'
SEARCH = ['/kaggle/input']
if DRIVE_ZIP_LINK:
    subprocess.run(['pip', '-q', 'install', 'gdown'], check=True)
    import gdown
    os.makedirs('/tmp/_in', exist_ok=True)
    if not os.path.exists('/tmp/fyp_kaggle.zip'):
        gdown.download(url=DRIVE_ZIP_LINK, output='/tmp/fyp_kaggle.zip', fuzzy=True, quiet=False)
    if not glob.glob('/tmp/_in/**/subset.csv', recursive=True):
        subprocess.run(['unzip', '-q', '-o', '/tmp/fyp_kaggle.zip', '-d', '/tmp/_in'], check=True)
    SEARCH.insert(0, '/tmp/_in')

def find(pattern, dirs=False):
    hits = []
    for root in SEARCH:
        hits += [h for h in glob.glob(f'{root}/**/{pattern}', recursive=True) if os.path.isdir(h) == dirs]
    return sorted(hits)

os.makedirs(OUTP, exist_ok=True)
sub = find('subset.csv'); assert sub, 'subset.csv not found: check the zip link (option A) or Add Input (option B).'
shutil.copy(sub[0], f'{OUTP}/subset.csv')
import pandas as _pd
first = _pd.read_csv(sub[0]).utt.iloc[0]
flac = find(f'{first}.flac'); assert flac, f'audio subset not found (looked for {first}.flac)'
if not os.path.exists(f'{OUTP}/subset'):
    os.symlink(os.path.dirname(flac[0]), f'{OUTP}/subset')
ck = find('Best_LA_model_for_DF.pth'); assert ck, 'Best_LA_model_for_DF.pth not found'
os.makedirs(f'{OUTP}/checkpoints', exist_ok=True)
if not os.path.exists(f'{OUTP}/checkpoints/Best_LA_model_for_DF.pth'):
    os.symlink(ck[0], f'{OUTP}/checkpoints/Best_LA_model_for_DF.pth')
ckb = find('LA_model.pth')
if ckb and not os.path.exists(f'{OUTP}/checkpoints/LA_model.pth'):
    os.symlink(ckb[0], f'{OUTP}/checkpoints/LA_model.pth')
print('checkpoint B:', ckb[0] if ckb else 'NOT FOUND: add LA_model.pth to your Kaggle dataset, or B is skipped')
sc = [h for h in find('dev__clean.csv') if os.path.basename(os.path.dirname(h)) == 'scores' and '/run_' not in h]
assert sc, 'scores/dev__clean.csv not found (checkpoint A scores from weeks 1-2)'
shutil.copytree(os.path.dirname(sc[0]), f'{OUTP}/scores', dirs_exist_ok=True)
lay = find('layers_clean.npz'); assert lay, 'week2/inside_A/layers_clean.npz not found'
os.makedirs(f'{OUTP}/week2', exist_ok=True)
if not os.path.exists(f'{OUTP}/week2/inside_A'):
    os.symlink(os.path.dirname(lay[0]), f'{OUTP}/week2/inside_A')
w3 = [d for d in find('week3', dirs=True) if not d.startswith(OUTP)]
if w3:
    shutil.copytree(w3[0], f'{OUTP}/week3', dirs_exist_ok=True)
print('subset:', os.path.dirname(flac[0])); print('checkpoint:', ck[0]); print('scores:', os.path.dirname(sc[0]))
print('layers:', os.path.dirname(lay[0])); print('earlier week-3 results:', w3[0] if w3 else 'none (starting fresh)')

In [ ]:
#@title 1 · Settings (week 6)
import os, json, time, glob
CFG = dict(
    SEED=1234, BATCH=16,
    OUT='/kaggle/working/fyp_pilot',      # the week-1 folder
    WORK='/kaggle/working/work',
    RUN_DETECTORS=['A', 'B', 'AASIST'],          # A = Best_LA_model_for_DF.pth, B = LA_model.pth, AASIST = Jung et al.
    INSIDE_CKPTS=['A'],                          # steps 7-9; add 'B' only if you have spare time
    INSIDE_LAYERS=[5, 11, 17, 23],               # layers (0-23) used for the test-time fix in step 9
    RESCORE_FOR_LLR=True,                        # re-score week-1 stages once to also save the 'fake' logit
    # Optional: folder in your Drive with the Mendeley ElevenLabs/Respeecher clips (unzipped). '' = skip.
    ELEVEN_DIR='',
    # ---- week 3 ----
    CAND_LAYERS=[3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23],   # candidate layers for the fit-half choice
    GATE_LAYER=3,                                             # early layer used by the phone-call gate
    N_BOOT=1000,                                              # bootstrap resamples for 95% intervals
    DO_2021=True,                                             # step 13: ASVspoof 2021 LA (about 7.6 GB download)
    N21=120,
    # ---- week 4 ----
    W4_MODELS=['A', 'B', 'SLS'],          # SSL-AASIST checkpoints A and B, and XLSR-SLS (Zhang et al., ACM MM 2024)
    SPLIT_LAYERS=[11, 17],                # layers for the axis-split test (W4-4)
    N_RAND=5,
    # ---- week 5 ----
    TRANSPLANT_LAYERS=[11, 17, 21, 23],   # W5-2: layers where each detector's drift is added / removed
    ATTACK_LAYERS=[11, 17, 23],           # W5-4: layers for the nearest-attack analysis
    SPLIT_SEEDS=[1, 2, 3, 4, 5],          # W5-5: extra speaker splits
    MS_LAYERS=[5, 9, 13, 17, 21],
    # ---- week 6 ----
    RTC_REPO='JunXueTech/RTCFake',
    N_GEN=800,                            # English genuine offline/online pairs to use (all if fewer)
    N_SPF=800,                            # English spoofed pairs (for EER and spoof drift)
)
for d in (CFG['WORK'], f"{CFG['OUT']}/figures", f"{CFG['OUT']}/week2"):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{CFG['OUT']}/subset.csv"), 'Week-1 folder not found: run the week-1 notebook first, or fix CFG["OUT"].'
print(json.dumps(CFG, indent=1))

In [ ]:
#@title 2 · Install packages and codecs
!pip -q install transformers soundfile pandas matplotlib scikit-learn scipy webrtc-noise-gain huggingface_hub
!apt-get -qq install -y opus-tools > /dev/null
import subprocess, re, shutil
def encoders(ff='ffmpeg'):
    out = subprocess.run([ff, '-hide_banner', '-encoders'], capture_output=True, text=True).stdout
    return {n for n in ('libgsm', 'libopus', 'pcm_mulaw', 'libopencore_amrnb', 'libvo_amrwbenc') if re.search(rf'\s{n}\s', out)}
FFMPEG, enc = 'ffmpeg', encoders()
FFMPEG_GSM = 'ffmpeg' if 'libgsm' in enc else None          # Colab's own ffmpeg has GSM
if 'libopencore_amrnb' not in enc:                          # ... but not AMR: use the static build, as in week 1
    if not os.path.exists('/kaggle/working/ffs/ffmpeg'):
        !wget -q -O /kaggle/working/ffs.tar.xz https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz
        !mkdir -p /kaggle/working/ffs && tar -xf /kaggle/working/ffs.tar.xz -C /kaggle/working/ffs --strip-components=1
    if os.path.exists('/kaggle/working/ffs/ffmpeg') and 'libopencore_amrnb' in encoders('/kaggle/working/ffs/ffmpeg'):
        FFMPEG, enc = '/kaggle/working/ffs/ffmpeg', encoders('/kaggle/working/ffs/ffmpeg')
HAVE_OPUS_TOOLS = shutil.which('opusenc') is not None and shutil.which('opusdec') is not None
try:
    import webrtc_noise_gain; HAVE_WEBRTC = True
except Exception as e:
    HAVE_WEBRTC = False; print('WebRTC processing not available:', e)
print('ffmpeg:', FFMPEG, '| encoders:', sorted(enc), '| GSM via:', FFMPEG_GSM)
print('opus-tools:', HAVE_OPUS_TOOLS, '| WebRTC:', HAVE_WEBRTC, '| AMR-WB:', 'libvo_amrwbenc' in enc)
assert 'libopencore_amrnb' in enc, 'AMR-NB encoder missing: the static ffmpeg download failed. Rerun this cell.'

In [ ]:
#@title 3 · Code and checkpoints (SSL-AASIST from your Drive, AASIST from GitHub)
REPO = '/kaggle/working/SSL_Anti-spoofing'
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/TakHemlata/SSL_Anti-spoofing {REPO}
AREPO = '/kaggle/working/aasist'
if not os.path.exists(AREPO):
    !git clone -q --depth 1 https://github.com/clovaai/aasist {AREPO}
commit = lambda r: subprocess.run(['git', '-C', r, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
COMMITS = {'SSL_Anti-spoofing': commit(REPO), 'aasist': commit(AREPO)}
cks = sorted(glob.glob(f"{CFG['OUT']}/checkpoints/**/*.pth", recursive=True))
CK = {'A': next((p for p in cks if os.path.basename(p) == 'Best_LA_model_for_DF.pth'), None),
      'B': next((p for p in cks if os.path.basename(p) == 'LA_model.pth'), None)}
print('commits:', COMMITS)
print('SSL-AASIST checkpoints:', {k: (os.path.basename(v) if v else None) for k, v in CK.items()})
assert CK['A'], 'Best_LA_model_for_DF.pth not found under fyp_pilot/checkpoints (it was downloaded in week 1).'
if not CK['B'] and 'B' in CFG['RUN_DETECTORS']:
    print('LA_model.pth not found: checkpoint B will be skipped.'); CFG['RUN_DETECTORS'].remove('B')

In [ ]:
#@title 4 · The week-1 audio subset (from Drive)
import numpy as np, pandas as pd
SUB = f"{CFG['OUT']}/subset"
subset = pd.read_csv(f"{CFG['OUT']}/subset.csv")
subset['path'] = SUB + '/' + subset.utt + '.flac'
missing = [p for p in subset.path if not os.path.exists(p)]
assert not missing, f'{len(missing)} subset files missing, e.g. {missing[:3]}'
EV, DV = subset[subset.part == 'eval'].reset_index(drop=True), subset[subset.part == 'dev'].reset_index(drop=True)
print(subset.groupby(['part', 'label']).size().rename('n'))
print('eval speakers:', EV.speaker.nunique())

In [ ]:
%%writefile /kaggle/working/pilot_lib.py
# ---------------------------------------------------------------------------
# Pilot library: call-stage simulator, alignment, SSL-AASIST loader, metrics.
# Kept in one file so it can be tested outside Colab and pasted into the notebook.
# ---------------------------------------------------------------------------
import os, re, sys, types, json, zlib, shutil, tempfile, subprocess
import numpy as np
import soundfile as sf
from scipy.signal import butter, sosfiltfilt, correlate

SR = 16000
CUT = 64600          # SSL-AASIST input length (~4.04 s), as in Tak et al.'s code


# ----------------------------- audio helpers ------------------------------
def tak_pad(x, max_len=CUT):
    """Exactly Tak et al.'s pad(): truncate, or tile short clips."""
    if len(x) >= max_len:
        return x[:max_len]
    reps = int(max_len / len(x)) + 1
    return np.tile(x, reps)[:max_len]


def _run(cmd):
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if r.returncode != 0:
        raise RuntimeError(' '.join(cmd) + '\n' + r.stderr.decode(errors='ignore')[-800:])


def ffmpeg_roundtrip(x, enc_args, codec_sr=8000, ext='wav', ffmpeg='ffmpeg'):
    """16 kHz float -> (resample to codec_sr) -> encode -> decode -> 16 kHz float."""
    with tempfile.TemporaryDirectory() as d:
        inp, mid, out = f'{d}/in.wav', f'{d}/mid.{ext}', f'{d}/out.wav'
        sf.write(inp, np.clip(x, -1, 1), SR, subtype='PCM_16')
        _run([ffmpeg, '-y', '-loglevel', 'error', '-i', inp, '-ac', '1', '-ar', str(codec_sr)] + enc_args + [mid])
        _run([ffmpeg, '-y', '-loglevel', 'error', '-i', mid, '-ac', '1', '-ar', str(SR), '-c:a', 'pcm_f32le', out])
        y, _ = sf.read(out, dtype='float32')
    return y


def align_to(ref, y, max_lag=1600):
    """Remove codec delay: find the lag that best matches ref, shift y, trim/pad to len(ref)."""
    n = min(len(ref), len(y), 2 * SR)
    a, b = ref[:n].astype(np.float64), y[:n].astype(np.float64)
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        lag = 0
    else:
        c = correlate(b, a, mode='full', method='fft')
        lags = np.arange(-n + 1, n)
        keep = np.abs(lags) <= max_lag
        lag = int(lags[keep][np.argmax(c[keep])])
    if lag > 0:
        y = y[lag:]
    elif lag < 0:
        y = np.concatenate([np.zeros(-lag, dtype=y.dtype), y])
    if len(y) < len(ref):
        y = np.concatenate([y, np.zeros(len(ref) - len(y), dtype=y.dtype)])
    return y[:len(ref)].astype(np.float32), lag


def bandpass(x, lo=300, hi=3400, order=6):
    sos = butter(order, [lo, hi], btype='bandpass', fs=SR, output='sos')
    return sosfiltfilt(sos, x).astype(np.float32)   # zero-phase: no delay


def rms_normalise(x, target_dbfs=-26.0):
    rms = np.sqrt(np.mean(x ** 2)) + 1e-12
    y = x * (10 ** (target_dbfs / 20) / rms)
    peak = np.max(np.abs(y))
    return (y / peak * 0.99 if peak > 0.99 else y).astype(np.float32)


def frame_loss(x, utt_id, rate, seed, frame=320, fade=40):
    """Drop 20 ms frames at random (seeded per utterance) and conceal by repeating
    the previous frame, attenuated, with short cross-fades. A simplified packet-loss
    concealment, documented as such; not the G.711 Appendix I algorithm."""
    rng = np.random.default_rng((zlib.crc32(utt_id.encode()) ^ seed) & 0xFFFFFFFF)
    y = x.copy()
    n = len(x) // frame
    lost = rng.random(n) < rate
    ramp = np.linspace(0, 1, fade, dtype=np.float32)
    run = 0
    for i in range(1, n):
        s = i * frame
        if lost[i]:
            run += 1
            seg = y[s - frame:s] * (0.7 ** run)          # repeat the last frame, attenuated
            seg[:fade] = y[s - 1] * (1 - ramp) + seg[:fade] * ramp   # smooth join, no click
            y[s:s + frame] = seg
        else:
            if run:   # fade back in from the concealed frame
                y[s:s + fade] = y[s - fade:s] * (1 - ramp) + x[s:s + fade] * ramp
            run = 0
    return y, float(lost.mean())


# ----------------------------- conditions ---------------------------------
def available_encoders(ffmpeg='ffmpeg'):
    out = subprocess.run([ffmpeg, '-hide_banner', '-encoders'], stdout=subprocess.PIPE, stderr=subprocess.PIPE).stdout.decode()
    return {name for name in ('libgsm', 'libopus', 'pcm_mulaw', 'libopencore_amrnb', 'libvo_amrwbenc') if re.search(rf'\s{name}\s', out)}


def make_conditions(encoders, seed, ffmpeg='ffmpeg'):
    """Returns {name: (description, fn(x, utt_id) -> (y, info))}. Every output is
    16 kHz, the same length as the input and sample-aligned with it."""
    C = {}
    C['clean'] = ('No processing', lambda x, u: (x, {}))
    C['clean_rms'] = ('Clean, loudness-normalised to -26 dBFS (loudness/AGC control)',
                      lambda x, u: (rms_normalise(x), {}))
    C['bandpass_300_3400'] = ('Telephone band 300-3,400 Hz, zero-phase filter (band-limit control)',
                              lambda x, u: (bandpass(x), {}))

    def coded(args, ext, sr=8000):
        def fn(x, u):
            y, lag = align_to(x, ffmpeg_roundtrip(x, args, sr, ext, ffmpeg))
            return y, {'lag': lag}
        return fn

    C['nb_resample'] = ('16 -> 8 -> 16 kHz resampling only (control for all 8 kHz codecs)',
                        coded(['-c:a', 'pcm_s16le'], 'wav'))
    if 'pcm_mulaw' in encoders:
        C['g711u'] = ('G.711 mu-law, 64 kbps', coded(['-c:a', 'pcm_mulaw'], 'wav'))

        def g711_loss(x, u):
            y, info = C['g711u'][1](x, u)
            y, lost = frame_loss(y, u, 0.10, seed)
            return y, dict(info, lost=lost)
        C['g711u_loss10'] = ('G.711 mu-law with 10% 20 ms frame loss, repetition concealment', g711_loss)
    if 'libgsm' in encoders:
        C['gsm_fr'] = ('GSM full-rate, 13 kbps (2G mobile)', coded(['-c:a', 'libgsm'], 'gsm'))
    if 'libopencore_amrnb' in encoders:
        C['amrnb_12k2'] = ('AMR-NB 12.2 kbps (mobile)', coded(['-c:a', 'libopencore_amrnb', '-b:a', '12.2k'], 'amr'))
        C['amrnb_4k75'] = ('AMR-NB 4.75 kbps (mobile, low rate)', coded(['-c:a', 'libopencore_amrnb', '-b:a', '4.75k'], 'amr'))
    if 'libopus' in encoders:
        C['opus_nb_8k'] = ('Opus narrowband, 8 kbps, VoIP mode',
                           coded(['-c:a', 'libopus', '-b:a', '8k', '-application', 'voip'], 'opus'))
    return C


# ----------------------------- metrics ------------------------------------
def det_curve(tar, non):
    """Identical to Tak et al.'s compute_det_curve (target = bona fide)."""
    n = tar.size + non.size
    s = np.concatenate([tar, non]); l = np.concatenate([np.ones(tar.size), np.zeros(non.size)])
    idx = np.argsort(s, kind='mergesort'); l = l[idx]
    tsum = np.cumsum(l); nsum = non.size - (np.arange(1, n + 1) - tsum)
    frr = np.concatenate([[0], tsum / tar.size]); far = np.concatenate([[1], nsum / non.size])
    thr = np.concatenate([[s[idx[0]] - 0.001], s[idx]])
    return frr, far, thr


def eer(tar, non):
    frr, far, thr = det_curve(tar, non)
    i = np.argmin(np.abs(frr - far))
    return float(np.mean([frr[i], far[i]])), float(thr[i])


def rates_at(tar, non, t):
    """Bona fide rejected (false alarm on a real caller) and spoof accepted (miss), at threshold t.
    A score >= t is accepted as bona fide."""
    return float(np.mean(tar < t)), float(np.mean(non >= t))


def dcf(p_fa_bona, p_miss_spoof, p_spoof, c_rej_bona=1.0, c_acc_spoof=10.0):
    """Normalised detection cost. 'Rejecting bona fide' and 'accepting spoof' are the two errors.
    Defaults follow ASVspoof 5 (C=1 and 10, P_spoof=0.05)."""
    cost = c_rej_bona * (1 - p_spoof) * p_fa_bona + c_acc_spoof * p_spoof * p_miss_spoof
    return float(cost / min(c_rej_bona * (1 - p_spoof), c_acc_spoof * p_spoof))


def min_dcf(tar, non, p_spoof, c1=1.0, c2=10.0):
    frr, far, _ = det_curve(tar, non)
    return float(min(dcf(fr, fa, p_spoof, c1, c2) for fr, fa in zip(frr, far)))


def boot_ci(d, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    m = [rng.choice(d, d.size, replace=True).mean() for _ in range(n)]
    return float(np.mean(d)), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


PRIORS = {'asv5': (0.05, 1.0, 10.0), 'balanced': (0.5, 1.0, 1.0)}


def summarise(scores, clean, thr):
    """scores/clean: DataFrames indexed by utt with columns score, label(1=bona fide), correct_clean."""
    out = {}
    for subset, mask in (('all', np.ones(len(scores), bool)), ('clean_correct', scores['correct_clean'].values)):
        s = scores[mask]; c = clean.loc[s.index]
        tar, non = s[s.label == 1].score.values, s[s.label == 0].score.values
        if tar.size < 2 or non.size < 2:          # e.g. an empty clean-correct subset
            out[subset] = {'eer': np.nan, 'fa_bona_at_thr': np.nan, 'miss_spoof_at_thr': np.nan,
                           'n_bona': int(tar.size), 'n_spoof': int(non.size)}
            continue
        e, _ = eer(tar, non)
        fa, miss = rates_at(tar, non, thr)
        r = {'eer': e, 'fa_bona_at_thr': fa, 'miss_spoof_at_thr': miss, 'hter_at_thr': (fa + miss) / 2}
        for k, (p, c1, c2) in PRIORS.items():
            r[f'minDCF_{k}'] = min_dcf(tar, non, p, c1, c2)
            r[f'actDCF_{k}'] = dcf(fa, miss, p, c1, c2)
            r[f'calib_loss_{k}'] = r[f'actDCF_{k}'] - r[f'minDCF_{k}']
        for lab, name in ((1, 'bona'), (0, 'spoof')):
            d = (s[s.label == lab].score.values - c[c.label == lab].score.values)
            m, lo, hi = boot_ci(d)
            r[f'shift_{name}_mean'], r[f'shift_{name}_lo'], r[f'shift_{name}_hi'] = m, lo, hi
        r['n_bona'], r['n_spoof'] = int(tar.size), int(non.size)
        out[subset] = r
    return out


def decision(row_clean, row):
    """Pre-registered pilot rule: clean EER <= 5% AND (EER rises >= 10 points OR
    real callers rejected at the transferred threshold rise >= 15 points)."""
    ok_clean = row_clean['eer'] <= 0.05
    d_eer = row['eer'] - row_clean['eer']
    d_fa = row['fa_bona_at_thr'] - row_clean['fa_bona_at_thr']
    counts = ok_clean and (d_eer >= 0.10 or d_fa >= 0.15)
    if not counts:
        kind = 'no_effect'
    elif d_eer >= 0.10:
        kind = 'ranking_breaks'
    else:
        kind = 'threshold_only'
    return counts, kind, d_eer, d_fa


# ----------------------------- model loader -------------------------------
def xlsr_config(n_layers=24, conv_bias=True):
    from transformers import Wav2Vec2Config
    return Wav2Vec2Config(
        hidden_size=1024, num_hidden_layers=n_layers, num_attention_heads=16, intermediate_size=4096,
        hidden_act='gelu', hidden_dropout=0.0, activation_dropout=0.0, attention_dropout=0.0,
        feat_proj_dropout=0.0, final_dropout=0.0, layerdrop=0.0, layer_norm_eps=1e-5,
        feat_extract_norm='layer', feat_extract_activation='gelu', conv_dim=(512,) * 7,
        conv_stride=(5, 2, 2, 2, 2, 2, 2), conv_kernel=(10, 3, 3, 3, 3, 2, 2), conv_bias=conv_bias,
        num_conv_pos_embeddings=128, num_conv_pos_embedding_groups=16, do_stable_layer_norm=True,
        apply_spec_augment=False, mask_time_prob=0.0, mask_feature_prob=0.0)


IGNORED_FS_PREFIXES = ('quantizer.', 'project_q.', 'final_proj.', 'project_inp.', 'target_glu.', 'input_quantizer.', 'mask_emb')


def map_fairseq_to_hf(sd_fs, hf_keys):
    """Rename fairseq wav2vec 2.0 parameter names to Hugging Face Wav2Vec2Model names.
    Module names checked against the fairseq commit bundled in Tak et al.'s repository."""
    hf_keys = set(hf_keys)
    param_wn = any('parametrizations.weight.original0' in k for k in hf_keys)
    out, unused = {}, []
    for k, v in sd_fs.items():
        nk = None
        if m := re.fullmatch(r'feature_extractor\.conv_layers\.(\d+)\.0\.(weight|bias)', k):
            nk = f'feature_extractor.conv_layers.{m[1]}.conv.{m[2]}'
        elif m := re.fullmatch(r'feature_extractor\.conv_layers\.(\d+)\.2\.1\.(weight|bias)', k):
            nk = f'feature_extractor.conv_layers.{m[1]}.layer_norm.{m[2]}'
        elif k in ('layer_norm.weight', 'layer_norm.bias'):
            nk = 'feature_projection.' + k
        elif m := re.fullmatch(r'post_extract_proj\.(weight|bias)', k):
            nk = f'feature_projection.projection.{m[1]}'
        elif k == 'encoder.pos_conv.0.bias':
            nk = 'encoder.pos_conv_embed.conv.bias'
        elif k == 'encoder.pos_conv.0.weight_g':
            nk = 'encoder.pos_conv_embed.conv.' + ('parametrizations.weight.original0' if param_wn else 'weight_g')
        elif k == 'encoder.pos_conv.0.weight_v':
            nk = 'encoder.pos_conv_embed.conv.' + ('parametrizations.weight.original1' if param_wn else 'weight_v')
        elif m := re.fullmatch(r'encoder\.layers\.(\d+)\.(.+)', k):
            rest = (m[2].replace('self_attn_layer_norm', 'layer_norm').replace('self_attn.', 'attention.')
                    .replace('fc1.', 'feed_forward.intermediate_dense.').replace('fc2.', 'feed_forward.output_dense.'))
            nk = f'encoder.layers.{m[1]}.{rest}'
        elif k in ('encoder.layer_norm.weight', 'encoder.layer_norm.bias'):
            nk = k
        elif k == 'mask_emb':
            nk = 'masked_spec_embed'
        if nk is None or nk not in hf_keys:
            unused.append(k)
        else:
            out[nk] = v
    missing = sorted(hf_keys - set(out))
    return out, unused, missing


def build_ssl_aasist(repo_dir, ckpt_path, device='cpu'):
    """Tak et al.'s SSL-AASIST with the fairseq front-end replaced by an identical
    Hugging Face Wav2Vec2Model, so no fairseq install is needed. Loads strictly."""
    import torch, torch.nn as nn
    from transformers import Wav2Vec2Model
    sd = torch.load(ckpt_path, map_location='cpu')
    if isinstance(sd, dict) and 'state_dict' in sd and isinstance(sd['state_dict'], dict):
        sd = sd['state_dict']
    sd = {k[7:] if k.startswith('module.') else k: v for k, v in sd.items()}
    pre = 'ssl_model.model.'
    fs = {k[len(pre):]: v for k, v in sd.items() if k.startswith(pre)}
    n_layers = 1 + max(int(m[1]) for k in fs if (m := re.match(r'encoder\.layers\.(\d+)\.', k)))
    conv_bias = 'feature_extractor.conv_layers.0.0.bias' in fs
    cfg = xlsr_config(n_layers, conv_bias)

    class HFSSL(nn.Module):
        def __init__(self, device):
            super().__init__()
            self.model = Wav2Vec2Model(cfg)
            self.out_dim = cfg.hidden_size
            self.device = device

        def extract_feat(self, x, output_hidden_states=False):
            if x.ndim == 3:
                x = x[:, :, 0]
            o = self.model(x, output_hidden_states=output_hidden_states)
            if output_hidden_states:
                self.last_hidden_states = o.hidden_states
            return o.last_hidden_state

    if 'fairseq' not in sys.modules:                 # model.py imports fairseq at the top; never used here
        sys.modules['fairseq'] = types.ModuleType('fairseq')
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    import model as tak
    tak.SSLModel = HFSSL
    net = tak.Model(None, device)
    hf_keys = list(net.ssl_model.model.state_dict().keys())
    mapped, unused, missing = map_fairseq_to_hf(fs, hf_keys)
    bad_unused = [k for k in unused if not k.startswith(IGNORED_FS_PREFIXES)]
    missing = [k for k in missing if k != 'masked_spec_embed']
    if bad_unused or missing:
        raise RuntimeError(f'Weight mapping failed.\nUnmapped checkpoint keys: {bad_unused[:20]}\nMissing model keys: {missing[:20]}')
    full = {k: v for k, v in sd.items() if not k.startswith(pre)}
    full.update({pre + k: v for k, v in mapped.items()})
    if 'masked_spec_embed' in hf_keys and 'masked_spec_embed' not in mapped:   # only exists in some versions
        full[pre + 'masked_spec_embed'] = net.ssl_model.model.masked_spec_embed.detach()
    net.load_state_dict(full, strict=True)
    report = {'xlsr_layers': n_layers, 'conv_bias': conv_bias, 'ssl_params_loaded': len(mapped),
              'ignored_pretraining_params': len(unused), 'backend_params_loaded': len(sd) - len(fs)}
    return net.to(device).eval(), report


# ===========================================================================
# Week 2 additions: shortcut controls, more realistic channels, a second
# architecture (AASIST) and hooks for looking inside the XLS-R front end.
# ===========================================================================
def trim_bounds(x, frame=320, rel_db=-35.0, margin=1600):
    """Start/end sample of the speech region: frames within rel_db of the loudest
    20 ms frame, padded by 100 ms. Computed on the CLEAN copy and applied to the
    call copy too, so the pair stays aligned (silence-shortcut control)."""
    n = max(1, len(x) // frame)
    e = np.array([np.mean(x[i * frame:(i + 1) * frame] ** 2) for i in range(n)]) + 1e-12
    db = 10 * np.log10(e / e.max())
    idx = np.where(db > rel_db)[0]
    if idx.size == 0:
        return 0, len(x)
    return max(0, idx[0] * frame - margin), min(len(x), (idx[-1] + 1) * frame + margin)


def opus_plc(x, loss_pct, bitrate_k=8, opusenc='opusenc', opusdec='opusdec', ffmpeg='ffmpeg'):
    """Opus narrowband speech at bitrate_k kbit/s with real Opus packet-loss concealment
    (opusdec --packet-loss). opusdec draws losses from an unseeded rand(), so the loss
    pattern is deterministic and identical for every file: documented, reproducible."""
    with tempfile.TemporaryDirectory() as d:
        sf.write(f'{d}/in.wav', np.clip(x, -1, 1), SR, subtype='PCM_16')
        _run([ffmpeg, '-y', '-loglevel', 'error', '-i', f'{d}/in.wav', '-ac', '1', '-ar', '8000', f'{d}/in8.wav'])
        _run([opusenc, '--quiet', '--bitrate', str(bitrate_k), '--speech', '--framesize', '20', f'{d}/in8.wav', f'{d}/a.opus'])
        _run([opusdec, '--quiet', '--rate', str(SR), '--packet-loss', str(loss_pct), f'{d}/a.opus', f'{d}/o.wav'])
        y, _ = sf.read(f'{d}/o.wav', dtype='float32')
    if y.ndim > 1:
        y = y[:, 0]
    return y


def webrtc_apm(x, noise_suppression=2, auto_gain_dbfs=3):
    """WebRTC noise suppression + automatic gain control (+ its high-pass filter), as phones
    and VoIP apps apply before encoding. 10 ms chunks of 16 kHz int16."""
    from webrtc_noise_gain import AudioProcessor
    ap = AudioProcessor(auto_gain_dbfs, noise_suppression)
    pcm = (np.clip(x, -1, 1) * 32767).astype(np.int16)
    n = len(pcm) // 160 * 160
    out = bytearray()
    for i in range(0, n, 160):
        out += ap.Process10ms(pcm[i:i + 160].tobytes()).audio
    y = np.frombuffer(bytes(out), dtype=np.int16).astype(np.float32) / 32768
    return np.concatenate([y, np.zeros(len(x) - len(y), np.float32)])


def _coded(args, ext, sr, ffmpeg):
    def fn(x, u):
        y, lag = align_to(x, ffmpeg_roundtrip(x, args, sr, ext, ffmpeg))
        return y, {'lag': lag}
    return fn


def frame_loss_ge(x, utt_id, mean_loss, mean_burst, seed, frame=320, fade=40):
    """Bursty 20 ms frame loss from a two-state Gilbert-Elliott chain (seeded per utterance), with the
    same repetition concealment as frame_loss. mean_loss = long-run loss rate, mean_burst = frames per burst."""
    rng = np.random.default_rng((zlib.crc32(utt_id.encode()) ^ (seed + 7)) & 0xFFFFFFFF)
    p_bg = 1.0 / mean_burst                       # bad -> good
    p_gb = mean_loss * p_bg / (1 - mean_loss)     # good -> bad
    n = len(x) // frame
    lost, bad = np.zeros(n, bool), False
    for i in range(n):
        bad = (rng.random() < p_gb) if not bad else (rng.random() >= p_bg)
        lost[i] = bad
    lost[0] = False
    y = x.copy(); ramp = np.linspace(0, 1, fade, dtype=np.float32); run = 0
    for i in range(1, n):
        s = i * frame
        if lost[i]:
            run += 1
            seg = y[s - frame:s] * (0.7 ** run)
            seg[:fade] = y[s - 1] * (1 - ramp) + seg[:fade] * ramp
            y[s:s + frame] = seg
        else:
            if run:
                y[s:s + fade] = y[s - fade:s] * (1 - ramp) + x[s:s + fade] * ramp
            run = 0
    return y, float(lost.mean())


def make_conditions_v2(encoders, seed, ffmpeg='ffmpeg', ffmpeg_gsm=None, have_opus_tools=False, have_webrtc=False):
    """Week-2 stages. Returns {name: (description, fn(x, utt_id) -> (y, info))}.
    Shortcut controls (names with __rms / __trim) are added later, per damaging stage."""
    base = make_conditions(encoders, seed, ffmpeg)
    C = {}
    if ffmpeg_gsm and 'libgsm' in available_encoders(ffmpeg_gsm):
        def gsm(x, u):
            y, lag = align_to(x, ffmpeg_roundtrip(x, ['-c:a', 'libgsm'], 8000, 'gsm', ffmpeg_gsm))
            return y, {'lag': lag}
        C['gsm_fr'] = ('GSM full-rate, 13 kbit/s (2G mobile)', gsm)
    if have_opus_tools:
        for p in (5, 10, 20):
            def f(x, u, p=p):
                y, lag = align_to(x, opus_plc(x, p, ffmpeg=ffmpeg))
                return y, {'lag': lag, 'lost': p / 100}
            C[f'opus_nb_8k_plc{p}'] = (f'Opus 8 kbit/s, {p}% packet loss, native Opus concealment', f)
    if have_webrtc and 'amrnb_12k2' in base:
        def apm_amr(x, u):
            w, lag = align_to(x, webrtc_apm(x))          # WebRTC adds a short delay: remove it first
            y, info = base['amrnb_12k2'][1](w, u)
            return y, dict(info, lag=lag + info.get('lag', 0))
        C['webrtc_amrnb_12k2'] = ('WebRTC noise suppression + gain control, then AMR-NB 12.2 kbit/s', apm_amr)
        def apm(x, u):
            y, lag = align_to(x, webrtc_apm(x))
            return y, {'lag': lag}
        C['webrtc_only'] = ('WebRTC noise suppression + gain control only', apm)
    if 'libopencore_amrnb' in encoders:                      # dose-response over AMR-NB rates
        for r in ('5.9', '7.4'):
            C[f"amrnb_{r.replace('.', 'k')}"] = (f'AMR-NB {r} kbit/s (mobile)', _coded(['-c:a', 'libopencore_amrnb', '-b:a', f'{r}k'], 'amr', 8000, ffmpeg))
    if 'libvo_amrwbenc' in encoders:                         # wideband (HD voice / VoLTE)
        for r in ('12.65', '23.85'):
            C[f"amrwb_{r.replace('.', 'k')}"] = (f'AMR-WB {r} kbit/s (HD voice, 16 kHz)', _coded(['-c:a', 'libvo_amrwbenc', '-b:a', f'{r}k'], 'amr', 16000, ffmpeg))
    if 'libopus' in encoders:                                # wideband VoIP, as WebRTC apps normally run
        for r in (16, 24):
            C[f'opus_wb_{r}k'] = (f'Opus wideband {r} kbit/s, VoIP mode', _coded(['-c:a', 'libopus', '-b:a', f'{r}k', '-application', 'voip'], 'opus', 16000, ffmpeg))
    if 'g711u' in base:                                      # bursty (Gilbert-Elliott) loss, mean burst 3 frames
        for m in (5, 10):
            def ge(x, u, m=m):
                y, info = base['g711u'][1](x, u)
                y, lost = frame_loss_ge(y, u, m / 100, 3.0, seed)
                return y, dict(info, lost=lost)
            C[f'g711u_burst{m}'] = (f'G.711 with {m}% bursty frame loss (Gilbert-Elliott, mean burst 60 ms)', ge)
    if 'amrnb_12k2' in base:
        def chain(x, u):
            y, info = base['amrnb_12k2'][1](bandpass(x), u)
            y, lost = frame_loss(y, u, 0.05, seed)
            return y, dict(info, lost=lost)
        C['chain_bp_amr12k2_loss5'] = ('Band-limit, AMR-NB 12.2 kbit/s, 5% frame loss', chain)
    return base, C


def shortcut_variants(stage_fn):
    """Loudness and silence controls applied AFTER the call stage (B in the mechanism slide).
    __rms : call copy loudness-normalised to -26 dBFS.
    __trim: clean-defined speech region cut from the call copy (leading/trailing silence removed)."""
    def rms(x, u):
        y, info = stage_fn(x, u)
        return rms_normalise(y), info
    def trim(x, u):
        y, info = stage_fn(x, u)
        s, e = trim_bounds(x)
        return y[s:e], dict(info, trim=(s, e))
    return rms, trim


def clean_trim(x, u):
    s, e = trim_bounds(x)
    return x[s:e], {'trim': (s, e)}


# ----------------------------- second architecture ------------------------
def build_aasist(repo_dir, device='cpu', variant='AASIST'):
    """Jung et al.'s released AASIST (raw-waveform, no SSL front end), trained on ASVspoof 2019 LA.
    Wrapped so model(x) returns logits like SSL-AASIST: column 1 = bona fide."""
    import torch, torch.nn as nn, importlib
    conf = json.load(open(f'{repo_dir}/config/{variant}.conf'))
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    mod = importlib.import_module('models.' + conf['model_config']['architecture'])
    net = mod.Model(conf['model_config'])
    sd = torch.load(f'{repo_dir}/models/weights/{variant}.pth', map_location='cpu')
    net.load_state_dict(sd, strict=True)

    class Wrap(nn.Module):
        def __init__(self, m):
            super().__init__(); self.m = m
        def forward(self, x):
            return self.m(x)[1]
    n_params = sum(p.numel() for p in net.parameters())
    return Wrap(net).to(device).eval(), {'arch': variant, 'params': n_params, 'nb_samp': conf['model_config']['nb_samp']}


# ----------------------------- inside the model ---------------------------
def xlsr_layers(net):
    """The 24 transformer layers of the (Hugging Face) XLS-R front end inside SSL-AASIST."""
    return net.ssl_model.model.encoder.layers


def _hs(out):
    return out[0] if isinstance(out, tuple) else out


def _replace(out, h):
    return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h


class LayerTaps:
    """Context manager: records the time-averaged output of every chosen layer (B x 1024 each)."""
    def __init__(self, net, layers):
        self.layers, self.store, self.handles = list(layers), {}, []
        self.mods = xlsr_layers(net)
    def __enter__(self):
        for l in self.layers:
            def hook(m, i, o, l=l):
                self.store[l] = _hs(o).float().mean(1).detach().cpu().numpy()
            self.handles.append(self.mods[l].register_forward_hook(hook))
        return self
    def __exit__(self, *a):
        for h in self.handles:
            h.remove()


class LayerShift:
    """Context manager: adds a fixed vector to every frame of one layer's output
    (test-time steering). vec: 1-D array of size 1024 (use -drift to remove a drift)."""
    def __init__(self, net, layer, vec):
        import torch
        self.mod = xlsr_layers(net)[layer]
        self.vec = torch.as_tensor(np.asarray(vec, np.float32))
        self.h = None
    def __enter__(self):
        def hook(m, i, o):
            h = _hs(o)
            return _replace(o, h + self.vec.to(h.device, h.dtype))
        self.h = self.mod.register_forward_hook(hook)
        return self
    def __exit__(self, *a):
        self.h.remove()


def speaker_split(speakers, seed=0):
    """Deterministic 50/50 split by speaker: 'fit' (estimate directions/thresholds) vs 'test'."""
    sp = sorted(set(speakers))
    rng = np.random.default_rng(seed); rng.shuffle(sp)
    fit = set(sp[:len(sp) // 2])
    return np.array(['fit' if s in fit else 'test' for s in speakers])



class LayerAffine:
    """Context manager: replaces one layer's output h (every frame) by h @ A + b (label-free alignment)."""
    def __init__(self, net, layer, A, b):
        import torch
        self.mod = xlsr_layers(net)[layer]
        self.A = torch.as_tensor(np.asarray(A, np.float32)); self.b = torch.as_tensor(np.asarray(b, np.float32))
        self.h = None
    def __enter__(self):
        def hook(m, i, o):
            h = _hs(o)
            return _replace(o, h @ self.A.to(h.device, h.dtype) + self.b.to(h.device, h.dtype))
        self.h = self.mod.register_forward_hook(hook)
        return self
    def __exit__(self, *a):
        self.h.remove()


def coral_affine(src, tgt, reg=0.1):
    """CORAL (Sun et al., 2016) as an affine map from the call-copy statistics (src) to the clean statistics
    (tgt), both N x D utterance means from the fit half. Returns A (D x D) and b (D) with x' = x @ A + b.
    reg = shrinkage towards the diagonal, since N is close to D."""
    def shrink(C):
        return (1 - reg) * C + reg * np.diag(np.diag(C)) + 1e-6 * np.eye(len(C))
    def msqrt(C, inv=False):
        w, V = np.linalg.eigh(C); w = np.clip(w, 1e-8, None)
        return (V * (w ** (-0.5 if inv else 0.5))) @ V.T
    ms, mt = src.mean(0), tgt.mean(0)
    Cs, Ct = shrink(np.cov(src.T)), shrink(np.cov(tgt.T))
    A = msqrt(Cs, inv=True) @ msqrt(Ct)
    return A.astype(np.float32), (mt - ms @ A).astype(np.float32)

In [ ]:
#@title 6 · Stages, detectors, thresholds and the scorer (resumes from Drive)
import sys, importlib, torch, soundfile as sf, gc
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, '/kaggle/working'); import pilot_lib as P; importlib.reload(P)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'Switch the runtime to a T4 GPU first.'

BASE, NEW = P.make_conditions_v2(enc, CFG['SEED'], FFMPEG, FFMPEG_GSM, HAVE_OPUS_TOOLS, HAVE_WEBRTC)
DAMAGING = [c for c in ('g711u_loss10', 'amrnb_4k75', 'opus_nb_8k') if c in BASE]
SHORT = {'clean_trim': ('Clean, leading/trailing silence removed (reference for __trim)', P.clean_trim)}
for c in DAMAGING:
    r, t = P.shortcut_variants(BASE[c][1])
    SHORT[f'{c}__rms'] = (f'{c}, then loudness-normalised to -26 dBFS', r)
    SHORT[f'{c}__trim'] = (f'{c}, then leading/trailing silence removed', t)
ALL = {**BASE, **NEW, **SHORT}
# check every stage on 2 s of noise; drop any whose codec is missing in this ffmpeg build
_x = (0.05 * np.random.default_rng(0).standard_normal(32000)).astype(np.float32)
for k in list(ALL):
    try:
        y, _ = ALL[k][1](_x, 'probe'); assert len(y) > 0
    except Exception as e:
        print(f'  dropping {k}: {str(e)[:120]}'); ALL.pop(k)
print(f'{len(ALL)} stages:'); [print(f'  {k:26s} {v[0]}') for k, v in ALL.items()]

FOLDER = {'A': f"{CFG['OUT']}/scores", 'B': f"{CFG['OUT']}/run_LA_model/scores", 'AASIST': f"{CFG['OUT']}/run_AASIST/scores"}
NAME = {'A': 'SSL-AASIST A (DF-tuned RawBoost)', 'B': 'SSL-AASIST B (LA-tuned RawBoost)', 'AASIST': 'AASIST (no SSL front end)'}
for f in FOLDER.values(): os.makedirs(f, exist_ok=True)
LOAD = {}

def load_detector(k):
    m, rep = P.build_aasist(AREPO, dev) if k == 'AASIST' else P.build_ssl_aasist(REPO, CK[k], dev)
    LOAD[k] = rep
    return m

def prep(row, cond):
    x, sr = sf.read(row.path, dtype='float32'); assert sr == P.SR
    y, info = ALL[cond][1](x, row.utt)
    return P.tak_pad(y), info

@torch.no_grad()
def score_set(model, df, cond, folder):
    out = f'{folder}/{df.part.iloc[0]}__{cond}.csv'
    if os.path.exists(out):
        d = pd.read_csv(out)
        if 'logit0' in d.columns or not CFG['RESCORE_FOR_LLR'] or model is None:
            return d
    t0, rows = time.time(), []
    with ThreadPoolExecutor(4) as ex:
        items = list(df.itertuples())
        for i in range(0, len(items), CFG['BATCH']):
            chunk = items[i:i + CFG['BATCH']]
            res = list(ex.map(lambda r: prep(r, cond), chunk))
            o = model(torch.tensor(np.stack([r[0] for r in res])).to(dev)).float().cpu().numpy()
            for r, (_, info), v in zip(chunk, res, o):
                rows.append(dict(utt=r.utt, speaker=r.speaker, attack=r.attack, label=r.label, cond=cond,
                                 score=float(v[1]), logit0=float(v[0]), lag=info.get('lag', 0), lost=info.get('lost', 0.0)))
    d = pd.DataFrame(rows); d.to_csv(out, index=False)
    print(f'  {df.part.iloc[0]:4s} {cond:26s} {len(d)} utts in {time.time()-t0:.0f}s')
    return d

TH = {}
def thresholds(k, model=None):
    # dev clean scores set every threshold: dev EER point (main), centre of the clean gap, 1% of dev fakes accepted
    dv = score_set(model, DV, 'clean', FOLDER[k]) if model is not None else pd.read_csv(f'{FOLDER[k]}/dev__clean.csv')
    b, s = dv[dv.label == 1].score.values, dv[dv.label == 0].score.values
    TH[k] = {'dev_EER': P.eer(b, s)[1], 'mid_gap': (np.percentile(b, 1) + np.percentile(s, 99)) / 2, 'fakes_1pct': np.percentile(s, 99)}
    return P.eer(b, s)[0], TH[k]

In [ ]:
#@title 7 · Inside the model: record the 24 XLS-R layers, clean vs call (about 10 min per checkpoint)
LAYERS_ALL = list(range(24))
SPLIT = P.speaker_split(EV.speaker.values, CFG['SEED'])        # half the speakers fit, the other half test (speaker-disjoint)
print('fit / test utterances:', (SPLIT == 'fit').sum(), (SPLIT == 'test').sum())
INSIDE_CONDS = ['clean'] + DAMAGING
WAV = {}                                                        # processed audio per half, reused by step 9

def waves(cond, part='test'):
    if (cond, part) not in WAV:
        idx = np.where(SPLIT == part)[0]
        with ThreadPoolExecutor(4) as ex:
            WAV[(cond, part)] = np.stack(list(ex.map(lambda i: prep(EV.iloc[i], cond)[0], idx))).astype(np.float32)
    return WAV[(cond, part)]

@torch.no_grad()
def record(model, cond, path):
    if os.path.exists(path):
        z = np.load(path); return z['H'].astype(np.float32), z['score']
    H, S = np.zeros((len(EV), 24, 1024), np.float16), np.zeros(len(EV), np.float32)
    with ThreadPoolExecutor(4) as ex:
        for i in range(0, len(EV), CFG['BATCH']):
            rows = [EV.iloc[j] for j in range(i, min(i + CFG['BATCH'], len(EV)))]
            x = np.stack([r[0] for r in ex.map(lambda r: prep(r, cond), rows)])
            with P.LayerTaps(model, LAYERS_ALL) as tp:
                s = model(torch.tensor(x).to(dev))[:, 1].float().cpu().numpy()
            H[i:i + len(rows)] = np.stack([tp.store[l] for l in LAYERS_ALL], 1).astype(np.float16)
            S[i:i + len(rows)] = s
    np.savez(path, H=H, score=S, utt=EV.utt.values)
    return H.astype(np.float32), S

INSIDE = {}
for k in CFG['INSIDE_CKPTS']:
    d = f"{CFG['OUT']}/week2/inside_{k}"; os.makedirs(d, exist_ok=True)
    need = [c for c in INSIDE_CONDS if not os.path.exists(f'{d}/layers_{c}.npz')] or not os.path.exists(f'{FOLDER[k]}/dev__clean.csv')
    model = load_detector(k) if need else None
    thresholds(k, model)
    INSIDE[k] = {}
    for c in INSIDE_CONDS:
        t0 = time.time(); INSIDE[k][c] = record(model, c, f'{d}/layers_{c}.npz'); print(f'{k} {c:14s} recorded in {time.time()-t0:.0f}s')
    if model is not None:
        del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
#@title 8 · Week-3 helpers: per-utterance scores saved to Drive, bootstrap intervals
W3 = f"{CFG['OUT']}/week3"; os.makedirs(f'{W3}/scores', exist_ok=True)
y = EV.label.values; fit, test = SPLIT == 'fit', SPLIT == 'test'
K = 'A'; THR = TH[K]['dev_EER']
Hc, Sc = INSIDE[K]['clean']
_M = {}
def model_A():
    if 'm' not in _M:
        _M['m'] = load_detector(K)
    return _M['m']

@torch.no_grad()
def scores_for(tag, cond, part, ctx_fn=None):
    # scores of the `part` half of `cond` audio, with an optional layer edit; cached per tag on Drive
    f = f'{W3}/scores/{tag}__{part}.npy'
    if os.path.exists(f):
        return np.load(f)
    m, X, out = model_A(), waves(cond, part), []
    for i in range(0, len(X), CFG['BATCH']):
        x = torch.tensor(X[i:i + CFG['BATCH']]).to(dev)
        if ctx_fn is None:
            s = m(x)
        else:
            with ctx_fn(m):
                s = m(x)
        out.append(s[:, 1].float().cpu().numpy())
    s = np.concatenate(out); np.save(f, s)
    return s

def eer_(s, yy):
    return P.eer(s[yy == 1], s[yy == 0])[0]

YT = y[test]
_rng = np.random.default_rng(CFG['SEED'])
_g, _f = np.where(YT == 1)[0], np.where(YT == 0)[0]
BI = [np.concatenate([_rng.choice(_g, len(_g)), _rng.choice(_f, len(_f))]) for _ in range(CFG['N_BOOT'])]

def ci_eer(s, ref=None):
    # EER on the test half with a 95% stratified bootstrap interval; paired interval for the change against `ref`
    e = eer_(s, YT); bs = np.array([eer_(s[i], YT[i]) for i in BI])
    row = dict(eer=e, eer_lo=np.percentile(bs, 2.5), eer_hi=np.percentile(bs, 97.5))
    if ref is not None:
        d = np.array([eer_(s[i], YT[i]) - eer_(ref[i], YT[i]) for i in BI])
        row.update(d_eer=e - eer_(ref, YT), d_lo=np.percentile(d, 2.5), d_hi=np.percentile(d, 97.5))
    return row

def op(s, thr=None):
    thr = THR if thr is None else thr
    b, f = s[YT == 1], s[YT == 0]
    rr, fa = P.rates_at(b, f, thr)
    return dict(real_rejected=rr, fakes_accepted=fa, hter=(rr + fa) / 2, genuine_mean=float(b.mean()), fake_mean=float(f.mean()))

print('fit / test:', fit.sum(), test.sum(), '| clean dev-EER threshold:', round(float(THR), 3))

In [ ]:
#@title W4-1 · XLSR-SLS: download, port to Hugging Face (no fairseq), check it works on clean audio (about 10 min)
import torch.nn as nn, torch.nn.functional as F
from transformers import Wav2Vec2Model
W4 = f"{CFG['OUT']}/week4"
for d in ('', '/scores', '/drifts'):
    os.makedirs(W4 + d, exist_ok=True)

sls_ck = [h for h in glob.glob('/kaggle/input/**/MMpaper_model.pth', recursive=True)]
if sls_ck:
    SLS_PATH = sls_ck[0]
else:
    from huggingface_hub import hf_hub_download
    SLS_PATH = hf_hub_download('SpeechAntiSpoofingBenchmarks/XLSR-SLS', 'MMpaper_model.pth', local_dir='/kaggle/working/sls')
print('XLSR-SLS weights:', SLS_PATH)

class _SSL(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.model = Wav2Vec2Model(cfg)

class XLSRSLS(nn.Module):
    """Zhang et al.'s SLS back end: every XLS-R layer output (as fairseq layer_results) is time-pooled, gated by a
    learned sigmoid (fc0), the gated layers are summed, then BatchNorm, SELU, 3x3 max-pool, fc1, SELU, fc3, SELU,
    log-softmax. Index 1 = bona fide. Layer outputs are taken with forward hooks, so LayerTaps/LayerShift work as for A."""
    def __init__(self, cfg):
        super().__init__()
        self.ssl_model = _SSL(cfg)
        self.fc0 = nn.Linear(1024, 1)
        self.first_bn = nn.BatchNorm2d(1)
        self.fc1 = nn.Linear(22847, 1024)
        self.fc3 = nn.Linear(1024, 2)
        self.last_gates = None
    def forward(self, x):
        if x.ndim == 3:
            x = x[..., 0]
        outs = []
        hs = [l.register_forward_hook(lambda m, i, o: outs.append(o[0] if isinstance(o, tuple) else o))
              for l in self.ssl_model.model.encoder.layers]
        try:
            self.ssl_model.model(x)
        finally:
            for h in hs:
                h.remove()
        L = torch.stack(outs, 1)                               # B, 24, T, 1024
        g = torch.sigmoid(self.fc0(L.mean(2)))                 # B, 24, 1
        self.last_gates = g[..., 0].detach().float().cpu().numpy()
        f = (L * g[..., None]).sum(1, keepdim=True)            # B, 1, T, 1024
        f = F.max_pool2d(F.selu(self.first_bn(f)), (3, 3))
        f = F.selu(self.fc1(torch.flatten(f, 1)))
        return F.log_softmax(F.selu(self.fc3(f)), dim=1)

def build_sls(path, device):
    try:
        sd = torch.load(path, map_location='cpu')
    except Exception:
        sd = torch.load(path, map_location='cpu', weights_only=False)   # public benchmark checkpoint
    for key in ('state_dict', 'model'):
        if isinstance(sd, dict) and key in sd and isinstance(sd[key], dict):
            sd = sd[key]
    sd = {k[7:] if k.startswith('module.') else k: v for k, v in sd.items()}
    k0 = next(k for k in sd if k.endswith('encoder.layers.0.self_attn.k_proj.weight'))
    pre = k0[:-len('encoder.layers.0.self_attn.k_proj.weight')]
    fs = {k[len(pre):]: v for k, v in sd.items() if k.startswith(pre)}
    rest = {k: v for k, v in sd.items() if not k.startswith(pre)}
    n_layers = 1 + max(int(k.split('.')[2]) for k in fs if k.startswith('encoder.layers.'))
    net = XLSRSLS(P.xlsr_config(n_layers, 'feature_extractor.conv_layers.0.0.bias' in fs))
    mapped, unused, missing = P.map_fairseq_to_hf(fs, net.ssl_model.model.state_dict().keys())
    bad = [k for k in unused if not k.startswith(P.IGNORED_FS_PREFIXES)]
    missing = [k for k in missing if k != 'masked_spec_embed']
    assert not bad and not missing, f'XLS-R mapping failed: {bad[:10]} {missing[:10]}'
    # back end: match by name, else by shape
    want = {k: v.shape for k, v in net.state_dict().items() if not k.startswith('ssl_model.')}
    back = {}
    for k, v in rest.items():
        if k in want and tuple(v.shape) == tuple(want[k]):
            back[k] = v
    if len(back) < len([k for k in want if 'num_batches_tracked' not in k]):
        groups = {}
        for k, v in rest.items():
            groups.setdefault(k.rsplit('.', 1)[0], {})[k.rsplit('.', 1)[1]] = v
        for g, p in groups.items():
            w = p.get('weight')
            if w is not None and tuple(w.shape) == (1, 1024): tgt = 'fc0'
            elif w is not None and tuple(w.shape) == (1024, 22847): tgt = 'fc1'
            elif w is not None and tuple(w.shape) == (2, 1024): tgt = 'fc3'
            elif 'running_mean' in p and p['running_mean'].numel() == 1: tgt = 'first_bn'
            else: continue
            for n, v in p.items():
                back[f'{tgt}.{n}'] = v
    full = {'ssl_model.model.' + k: v for k, v in mapped.items()}
    if 'masked_spec_embed' in net.ssl_model.model.state_dict():
        full.setdefault('ssl_model.model.masked_spec_embed', net.ssl_model.model.masked_spec_embed.detach())
    full.update(back)
    res = net.load_state_dict(full, strict=False)
    miss = [k for k in res.missing_keys if 'num_batches_tracked' not in k]
    assert not miss, f'XLSR-SLS back end not loaded: {miss}. Back-end keys in the file: {[(k, tuple(v.shape)) for k, v in rest.items()]}'
    return net.to(device).eval()

SLS = build_sls(SLS_PATH, dev)

@torch.no_grad()
def scores_of(model, X):
    out = []
    for i in range(0, len(X), CFG['BATCH']):
        out.append(model(torch.tensor(X[i:i + CFG['BATCH']]).to(dev))[:, 1].float().cpu().numpy())
    return np.concatenate(out)

def audio_of(df, cond):
    with ThreadPoolExecutor(4) as ex:
        return np.stack(list(ex.map(lambda i: prep(df.iloc[i], cond)[0], range(len(df))))).astype(np.float32)

XDV = audio_of(DV, 'clean')
s_dv = scores_of(SLS, XDV)
b_, f_ = s_dv[DV.label.values == 1], s_dv[DV.label.values == 0]
TH['SLS'] = {'dev_EER': P.eer(b_, f_)[1]}
print(f"XLSR-SLS clean dev EER {100 * P.eer(b_, f_)[0]:.2f}% | threshold {TH['SLS']['dev_EER']:.3f}")
assert P.eer(b_, f_)[0] < 0.05, 'XLSR-SLS port looks wrong (clean dev EER >= 5%). Send this output to Claude.'

In [ ]:
#@title W6-1 · ASVspoof 2019 clean pass: each detector's clean real-to-fake axis; week-4 curves (about 10 min)
import io, re, tarfile, zlib
from scipy.optimize import curve_fit
from scipy.stats import spearmanr
from scipy.signal import resample_poly
W6 = f"{CFG['OUT']}/week6"; os.makedirs(W6, exist_ok=True)
MODELS = {'A': load_detector('A')}
if CK.get('B'):
    MODELS['B'] = load_detector('B')
    if 'B' not in TH:
        thresholds('B', MODELS['B'])
MODELS['SLS'] = SLS
LAYERS_ALL = list(range(24))
gen, spf = y == 1, y == 0

@torch.no_grad()
def run_models(X, names=None):
    names = names or list(MODELS)
    res = {k: ([], [], []) for k in names}
    for i in range(0, len(X), CFG['BATCH']):
        xb = torch.tensor(X[i:i + CFG['BATCH']]).to(dev)
        for k in names:
            m = MODELS[k]
            with P.LayerTaps(m, LAYERS_ALL) as tp:
                s = m(xb)[:, 1].float().cpu().numpy()
            res[k][0].append(np.stack([tp.store[l] for l in LAYERS_ALL], 1).astype(np.float16))
            res[k][1].append(s)
            if k == 'SLS':
                res[k][2].append(m.last_gates)
    return {k: (np.concatenate(a), np.concatenate(b), np.concatenate(c) if c else None) for k, (a, b, c) in res.items()}

@torch.no_grad()
def score_model(m, X, ctx=None):
    out = []
    for i in range(0, len(X), CFG['BATCH']):
        x = torch.tensor(X[i:i + CFG['BATCH']]).to(dev)
        if ctx is None:
            s = m(x)
        else:
            with ctx(m):
                s = m(x)
        out.append(s[:, 1].float().cpu().numpy())
    return np.concatenate(out)

f_clean = {k: f'{W6}/asv_clean_{k}.npz' for k in MODELS}
if all(os.path.exists(f) for f in f_clean.values()):
    R0 = {k: (np.load(f)['H'], np.load(f)['score'], np.load(f)['gates']) for k, f in f_clean.items()}
else:
    R0 = run_models(audio_of(EV, 'clean'))
    for k, (Hk, s, g) in R0.items():
        np.savez(f_clean[k], H=Hk, score=s, gates=g if g is not None else np.zeros(0))
U = {k: R0[k][0][fit & spf].astype(np.float32).mean(0) - R0[k][0][fit & gen].astype(np.float32).mean(0) for k in MODELS}
GCLEAN = R0['SLS'][2].mean(0)
W_READ = {'A': np.eye(24)[23], 'B': np.eye(24)[23], 'SLS': GCLEAN / GCLEAN.sum()}
del R0; gc.collect()

W4T = pd.read_csv(io.StringIO("""
stage,model,l17,ro,rej
amrnb_12k2,A,0.183,0.305,0.015
amrnb_4k75,A,0.823,1.017,0.871
amrnb_5k9,A,0.62,0.922,0.645
amrnb_7k4,A,0.396,0.682,0.202
amrwb_12k65,A,0.345,0.618,0.129
amrwb_23k85,A,0.153,0.274,0.009
bandpass_300_3400,A,0.231,0.366,0.018
chain_bp_amr12k2_loss5,A,0.699,0.936,0.616
g711u,A,0.042,0.051,0.0
g711u_burst10,A,0.419,0.661,0.208
g711u_burst5,A,0.257,0.415,0.061
g711u_loss10,A,0.614,0.895,0.599
gsm_fr,A,0.368,0.672,0.173
nb_resample,A,0.037,0.04,0.0
opus_nb_8k,A,0.451,0.755,0.309
opus_nb_8k_plc10,A,0.741,0.962,0.785
opus_nb_8k_plc20,A,0.885,1.004,0.925
opus_nb_8k_plc5,A,0.578,0.874,0.537
opus_wb_16k,A,0.099,0.156,0.0
opus_wb_24k,A,0.041,0.048,0.0
webrtc_amrnb_12k2,A,0.275,0.559,0.075
webrtc_only,A,0.055,0.153,0.0
amrnb_12k2,B,0.131,0.011,0.0
amrnb_4k75,B,0.725,0.538,0.386
amrnb_5k9,B,0.425,0.155,0.048
amrnb_7k4,B,0.265,0.048,0.004
amrwb_12k65,B,0.118,0.01,0.0
amrwb_23k85,B,0.058,0.003,0.0
bandpass_300_3400,B,0.101,-0.004,0.0
chain_bp_amr12k2_loss5,B,0.427,0.145,0.029
g711u,B,0.047,0.002,0.0
g711u_burst10,B,0.291,0.085,0.009
g711u_burst5,B,0.168,0.032,0.002
g711u_loss10,B,0.464,0.237,0.046
gsm_fr,B,0.24,0.043,0.004
nb_resample,B,0.045,0.002,0.0
opus_nb_8k,B,0.279,0.057,0.009
opus_nb_8k_plc10,B,0.636,0.423,0.246
opus_nb_8k_plc20,B,0.863,0.752,0.601
opus_nb_8k_plc5,B,0.407,0.142,0.039
opus_wb_16k,B,0.052,0.003,0.0
opus_wb_24k,B,0.025,0.001,0.0
webrtc_amrnb_12k2,B,0.211,0.051,0.002
webrtc_only,B,0.043,0.018,0.0
amrnb_12k2,SLS,0.101,0.164,0.009
amrnb_4k75,SLS,0.663,0.64,0.814
amrnb_5k9,SLS,0.444,0.501,0.476
amrnb_7k4,SLS,0.241,0.331,0.105
amrwb_12k65,SLS,0.218,0.286,0.099
amrwb_23k85,SLS,0.084,0.124,0.004
bandpass_300_3400,SLS,0.108,0.155,0.0
chain_bp_amr12k2_loss5,SLS,0.5,0.519,0.52
g711u,SLS,0.032,0.063,0.0
g711u_burst10,SLS,0.336,0.386,0.221
g711u_burst5,SLS,0.193,0.246,0.068
g711u_loss10,SLS,0.525,0.563,0.647
gsm_fr,SLS,0.247,0.331,0.145
nb_resample,SLS,0.028,0.055,0.0
opus_nb_8k,SLS,0.323,0.4,0.241
opus_nb_8k_plc10,SLS,0.617,0.611,0.743
opus_nb_8k_plc20,SLS,0.774,0.708,0.908
opus_nb_8k_plc5,SLS,0.45,0.499,0.476
opus_wb_16k,SLS,0.054,0.08,0.0
opus_wb_24k,SLS,0.021,0.032,0.0
webrtc_amrnb_12k2,SLS,0.168,0.269,0.031
webrtc_only,SLS,0.033,0.068,0.0
""".strip()))
SIG = lambda x, a, b: 1 / (1 + np.exp(-a * (x - b)))
CURVE = {(k, f): curve_fit(SIG, W4T[W4T.model == k][f], W4T[W4T.model == k].rej, p0=[8, .6], maxfev=20000)[0]
         for k in MODELS for f in ('l17', 'ro')}
print('thresholds:', {k: round(float(TH[k]['dev_EER']), 3) for k in MODELS})
print('curves (slope a, midpoint b) fixed from the 22 simulated stages:', {f'{k}/{f}': np.round(v, 3).tolist() for (k, f), v in CURVE.items()})

In [ ]:
#@title W6-2 · RTCFake dev: download, keep English pairs, score offline and online copies (about 40 min)
from huggingface_hub import hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN not found: Add-ons -> Secrets -> add HF_TOKEN and tick it for this notebook.'
RTC = '/kaggle/working/rtc'; os.makedirs(RTC, exist_ok=True)
get = lambda f: hf_hub_download(CFG['RTC_REPO'], f, repo_type='dataset', token=HF_TOKEN, local_dir=RTC)
lab_f, pair_f = get('data/dev_label.txt'), get('data/meta/dev_offline_online_pairs.csv')
LAB = {}
for line in open(lab_f):
    p = line.split()
    if len(p) >= 2:
        LAB[p[0].strip()] = p[-1].strip()
PR = pd.read_csv(pair_f).dropna()
PR['off'] = PR.offline_id.str.replace(r'^dev/', '', regex=True); PR['on'] = PR.online_id.str.replace(r'^dev/', '', regex=True)
PR = PR[PR.off.str.contains('/en/') & PR.on.str.contains('/en/')]
PR['label_off'] = PR.off.map(LAB); PR['label_on'] = PR.on.map(LAB)
print('English pairs:', len(PR), '| offline labels:', PR.label_off.value_counts(dropna=False).to_dict(),
      '| labels agree:', float((PR.label_off == PR.label_on).mean()))
PR = PR[PR.label_off.isin(['bonafide', 'spoof'])]
g_ = PR[PR.label_off == 'bonafide'].sample(min(CFG['N_GEN'], (PR.label_off == 'bonafide').sum()), random_state=CFG['SEED'])
s_ = PR[PR.label_off == 'spoof'].sample(min(CFG['N_SPF'], (PR.label_off == 'spoof').sum()), random_state=CFG['SEED'])
SELP = pd.concat([g_, s_]).reset_index(drop=True); SELP['y'] = (SELP.label_off == 'bonafide').astype(int)
print('selected pairs: genuine', SELP.y.sum(), '| spoof', (SELP.y == 0).sum())

AUD = f'{RTC}/audio'
want = set(SELP.off) | set(SELP.on)
have = {w for w in want if os.path.exists(f'{AUD}/{w}')}
if len(have) < len(want):
    tgz = get('data/wav/dev.tar.gz'); n = 0
    with tarfile.open(tgz, 'r|gz') as tf:
        for m in tf:
            if not m.isfile():
                continue
            key = '/'.join(m.name.split('/')[-3:])
            if key in want:
                os.makedirs(os.path.dirname(f'{AUD}/{key}'), exist_ok=True)
                with tf.extractfile(m) as src, open(f'{AUD}/{key}', 'wb') as dst:
                    shutil.copyfileobj(src, dst)
                n += 1
    print('extracted', n, 'of', len(want)); os.remove(tgz)
SELP = SELP[[os.path.exists(f'{AUD}/{a}') and os.path.exists(f'{AUD}/{b}') for a, b in zip(SELP.off, SELP.on)]].reset_index(drop=True)
print('pairs with both files:', len(SELP))

def load16(path):
    x, sr = sf.read(path, dtype='float32')
    if x.ndim > 1:
        x = x.mean(1)
    if sr != P.SR:
        from math import gcd
        g = gcd(sr, P.SR); x = resample_poly(x, P.SR // g, sr // g).astype(np.float32)
    return x, sr
SRS = set()
def pair_audio(r):
    a, sa = load16(f'{AUD}/{r.off}'); b, sb = load16(f'{AUD}/{r.on}'); SRS.update([sa, sb])
    b, lag = P.align_to(a, b, max_lag=8000)                  # up to 0.5 s of app delay
    return P.tak_pad(a), P.tak_pad(b), lag
with ThreadPoolExecutor(4) as ex:
    out = list(ex.map(pair_audio, SELP.itertuples()))
XOFF = np.stack([o[0] for o in out]).astype(np.float32); XON = np.stack([o[1] for o in out]).astype(np.float32)
SELP['lag'] = [o[2] for o in out]
print('source sample rates:', SRS, '| median |lag| (ms):', round(float(np.median(np.abs(SELP.lag))) / 16, 1))
REC6 = {}
for tag, X in (('off', XOFF), ('on', XON)):
    t0 = time.time(); R = run_models(X)
    for k, (Hk, s, g) in R.items():
        REC6[(k, tag)] = (Hk, s)
    print(tag, f'{time.time() - t0:.0f}s')
SC = pd.DataFrame({'off': SELP.off, 'on': SELP.on, 'y': SELP.y, 'lag': SELP.lag,
                   **{f'{k}_{t}': REC6[(k, t)][1] for k in MODELS for t in ('off', 'on')}})
SC.to_csv(f'{W6}/rtcfake_dev_pair_scores.csv', index=False)

In [ ]:
#@title W6-3 · Same push toward "fake"? Rejection, drift per layer, and prediction from the simulated curves
yr = SELP.y.values; G, Sp = yr == 1, yr == 0
ROWS, LAYROWS = [], []
for k in MODELS:
    thr = TH[k]['dev_EER']; Hoff, soff = REC6[(k, 'off')]; Hon, son = REC6[(k, 'on')]
    u = U[k]; uu = (u * u).sum(1)
    d = (Hon[G].astype(np.float32) - Hoff[G].astype(np.float32)).mean(0)
    ds = (Hon[Sp].astype(np.float32) - Hoff[Sp].astype(np.float32)).mean(0)
    pj, pjs = (d * u).sum(1) / uu, (ds * u).sum(1) / uu
    cs = (d * u).sum(1) / (np.linalg.norm(d, axis=1) * np.sqrt(uu) + 1e-12)
    sz = np.linalg.norm(d, axis=1) / np.sqrt(uu)
    for l in LAYERS_ALL:
        LAYROWS.append(dict(model=k, layer=l, proj_genuine=pj[l], proj_spoof=pjs[l], cos_genuine=cs[l], size_genuine=sz[l]))
    ro = float(pj @ W_READ[k])
    rr_off, fa_off = P.rates_at(soff[G], soff[Sp], thr); rr_on, fa_on = P.rates_at(son[G], son[Sp], thr)
    ROWS.append(dict(model=k, n_gen=int(G.sum()), n_spf=int(Sp.sum()),
                     eer_off=P.eer(soff[G], soff[Sp])[0], eer_on=P.eer(son[G], son[Sp])[0],
                     rej_off=rr_off, rej_on=rr_on, fakes_acc_off=fa_off, fakes_acc_on=fa_on,
                     genuine_shift=float((son - soff)[G].mean()), spoof_shift=float((son - soff)[Sp].mean()),
                     drift_l17=pj[17], drift_ro=ro, cos_l17=cs[17], spoof_drift_l17=pjs[17],
                     pred_rej_l17=SIG(pj[17], *CURVE[(k, 'l17')]), pred_rej_ro=SIG(ro, *CURVE[(k, 'ro')])))
RT = pd.DataFrame(ROWS); LY = pd.DataFrame(LAYROWS)
RT.to_csv(f'{W6}/rtcfake_summary.csv', index=False); LY.to_csv(f'{W6}/rtcfake_layers.csv', index=False)
display(RT.round(3))
display(LY.pivot(index='layer', columns='model', values='proj_genuine').round(3).T)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(8, 2.8))
for k in MODELS:
    t = LY[LY.model == k]; ax[0].plot(t.layer, t.proj_genuine, label=k); ax[1].plot(t.layer, t.proj_spoof, label=k)
for a, tt in zip(ax, ('Genuine', 'Spoofed')):
    a.axhline(0, color='#999', lw=.7); a.axhline(1, color='#999', ls='--', lw=.7); a.set_title(f'{tt}: RTCFake online vs offline'); a.set_xlabel('XLS-R layer')
ax[0].legend(); fig.tight_layout(); fig.savefig(f"{CFG['OUT']}/figures/week6_rtcfake_layers.png", dpi=200); plt.show()

In [ ]:
#@title W6-4 · Within the channel: do genuine utterances that drift more lose more score? (seconds)
WITHIN = []
for k in MODELS:
    Hoff, soff = REC6[(k, 'off')]; Hon, son = REC6[(k, 'on')]
    for L in (11, 17, 23):
        u = U[k][L]; per = ((Hon[G, L].astype(np.float32) - Hoff[G, L].astype(np.float32)) @ u) / (u @ u)
        drop = soff[G] - son[G]
        rho, p = spearmanr(per, drop)
        WITHIN.append(dict(model=k, layer=L, spearman_drift_vs_score_drop=rho, p=p, n=int(G.sum()),
                           drift_median=float(np.median(per)), drift_q90=float(np.quantile(per, .9))))
WITHIN = pd.DataFrame(WITHIN); WITHIN.to_csv(f'{W6}/rtcfake_within_channel.csv', index=False); display(WITHIN.round(3))

In [ ]:
#@title W6-5 · Correction on real transmission, checkpoint A: fit on half the pairs, remove at layer 17 on the other half (about 5 min)
rng6 = np.random.default_rng(CFG['SEED'] + 6)
half = rng6.permutation(len(SELP)) < len(SELP) // 2          # random halves of pairs (RTCFake dev lists no speaker IDs here)
HoffA, soffA = REC6[('A', 'off')]; HonA, sonA = REC6[('A', 'on')]
CORR = []
for L in (11, 17):
    v = (HonA[half, L].astype(np.float32) - HoffA[half, L].astype(np.float32)).mean(0)
    s = score_model(MODELS['A'], XON[~half], lambda m, v=v, L=L: P.LayerShift(m, L, -v))
    yy = yr[~half]; thr = TH['A']['dev_EER']
    for name, sc in (('offline', soffA[~half]), ('online, uncorrected', sonA[~half]), (f'online, drift removed at L{L}', s)):
        rr, fa = P.rates_at(sc[yy == 1], sc[yy == 0], thr)
        CORR.append(dict(layer=L, condition=name, eer=P.eer(sc[yy == 1], sc[yy == 0])[0], real_rejected=rr, fakes_accepted=fa))
CORR = pd.DataFrame(CORR).drop_duplicates('condition'); CORR.to_csv(f'{W6}/rtcfake_correction_A.csv', index=False); display(CORR.round(3))

In [ ]:
#@title W6-6 · Summary, and the zip for GitHub
import contextlib, zipfile
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
    print('WEEK 6 SUMMARY: RTCFake dev, English offline/online pairs')
    print('pairs used: genuine', int(G.sum()), '| spoof', int(Sp.sum()), '| source sample rates', SRS, '| median |lag| ms', round(float(np.median(np.abs(SELP.lag))) / 16, 1))
    print('curves (slope a, midpoint b):', {f'{k}/{f}': np.round(v, 3).tolist() for (k, f), v in CURVE.items()})
    print('\nOFFLINE vs ONLINE, rejection at each detector\'s clean ASVspoof threshold, drift on the ASVspoof axis, prediction from the simulated-channel curves:')
    print(RT.round(3).to_string(index=False))
    print('\nGENUINE DRIFT PER LAYER (projection on each detector\'s clean ASVspoof axis):')
    print(LY.pivot(index='model', columns='layer', values='proj_genuine').round(3).to_string())
    print('\nSPOOF DRIFT PER LAYER:'); print(LY.pivot(index='model', columns='layer', values='proj_spoof').round(3).to_string())
    print('\nCOSINE of genuine drift with the axis:'); print(LY.pivot(index='model', columns='layer', values='cos_genuine').round(3).to_string())
    print('\nWITHIN-CHANNEL (per genuine utterance: drift along the axis vs score drop):'); print(WITHIN.round(3).to_string(index=False))
    print('\nCORRECTION ON REAL TRANSMISSION, checkpoint A (fit on half the pairs, tested on the other half):'); print(CORR.round(3).to_string(index=False))
open(f'{W6}/week6_summary.txt', 'w').write(buf.getvalue()); print(buf.getvalue())
zp = '/kaggle/working/week6_for_github.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{W6}/*.csv') + glob.glob(f'{W6}/*.txt') + glob.glob(f"{CFG['OUT']}/figures/week6_*.png"):
        z.write(f, os.path.relpath(f, CFG['OUT']))
print(zp, os.path.getsize(zp) // 1024, 'KB')